In [2]:
import cv2
import numpy as np
import pandas as pd
import os
import shutil
import torch
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

![YOLO11](./YOLO11.png)

In [ ]:
def RLE_to_mask(rle,shape=(256, 1600)):
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths

    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order='F')

def mask_to_boxes(mask):
    mask = mask.astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        boxes.append([x, y, x + w, y + h])  # convert to x1,y1,x2,y2
    return boxes

def box_to_yolo(box, img_w, img_h):
    x1, y1, x2, y2 = box

    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    width = x2 - x1
    height = y2 - y1

    x = x_center / img_w
    y = y_center / img_h
    w = width / img_w
    h = height / img_h

    return (x, y, w, h)

Check CUDA

In [1]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

NameError: name 'torch' is not defined

Train/Test

In [ ]:
model = YOLO("yolov8n.pt")
results = model.train(data="yolo.yaml", epochs=10, imgsz=512, batch = 16, device="cuda:0")

Ultralytics 8.4.24  Python-3.10.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, pl

Acurracy metrics (y_true & y_pred are masks)

In [10]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def pixel_accuracy(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    return accuracy_score(y_true, y_pred)

In [ ]:
model = YOLO("best.pt") 

test_folder = "../data/kaggle_data/test_images"
test_images = [os.path.join(test_folder, f) for f in os.listdir(test_folder)]
test_images.sort() 


def mask_to_rle(mask):
    pixels = mask.T.flatten() 
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)



image 1/1 c:\Users\40757\Desktop\Steel-Defect-Detection\data\kaggle_data\test_images\0000f269f.jpg: 96x512 1 3, 43.5ms
Speed: 3.7ms preprocess, 43.5ms inference, 8.5ms postprocess per image at shape (1, 3, 96, 512)

image 1/1 c:\Users\40757\Desktop\Steel-Defect-Detection\data\kaggle_data\test_images\000ccc2ac.jpg: 96x512 (no detections), 12.0ms
Speed: 1.5ms preprocess, 12.0ms inference, 1.3ms postprocess per image at shape (1, 3, 96, 512)

image 1/1 c:\Users\40757\Desktop\Steel-Defect-Detection\data\kaggle_data\test_images\002451917.jpg: 96x512 1 3, 12.0ms
Speed: 1.4ms preprocess, 12.0ms inference, 2.5ms postprocess per image at shape (1, 3, 96, 512)

image 1/1 c:\Users\40757\Desktop\Steel-Defect-Detection\data\kaggle_data\test_images\003c5da97.jpg: 96x512 1 3, 12.1ms
Speed: 1.5ms preprocess, 12.1ms inference, 2.2ms postprocess per image at shape (1, 3, 96, 512)

image 1/1 c:\Users\40757\Desktop\Steel-Defect-Detection\data\kaggle_data\test_images\0042e163f.jpg: 96x512 (no detections),